In [1]:
# ==============================================================================
# KAGGLE CRAWLER FOR GIANT DOMAINS (120ask.com / familydoctor.com.cn)
# Tích hợp: Phân lô (Batching), Time Guard (Tự dừng an toàn sau 11h), Zstd Shards
# ==============================================================================

### Tóm tắt cách vận hành:
- Lần chạy 1: Đặt OFFSET = 0, BATCH_SIZE = 35000. Bấm Save & Run All, tắt máy.
- Lần chạy 2 (sau khi lần 1 xong hoặc chạy song song trên Notebook khác): Đặt OFFSET = 35000, BATCH_SIZE = 35000.

Bộ đếm Time Guard (10.8 tiếng) đảm bảo dù mạng có chậm đến mấy thì máy chủ Kaggle luôn dừng kịp để đóng gói zip ra tab Output, không bao giờ bị lỗi quá 12h!

In [2]:
# 1. CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT
%pip install -q trafilatura zstandard pyarrow datasets aiohttp protego nest_asyncio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import sys
import time
import asyncio
from pathlib import Path
from dataclasses import replace

# Tự động nạp thư mục src vào sys.path để import corpus_crawler
CRAWLER_DIR = Path("tasks/01_corpus_collection/crawler").resolve()
if (CRAWLER_DIR / "src").exists():
    sys.path.insert(0, str(CRAWLER_DIR / "src"))
elif Path("src").exists():
    sys.path.insert(0, str(Path("src").resolve()))

# Hỗ trợ chạy asyncio trực tiếp trong Jupyter Notebook mà không bị RuntimeError
import nest_asyncio
nest_asyncio.apply()

# ------------------------------------------------------------------------------
# 2. IMPORT CÁC MODULE VÀ HÀM CHÍNH TỪ SRC/CORPUS_CRAWLER
# ------------------------------------------------------------------------------
from corpus_crawler.settings import Settings, load_settings
from corpus_crawler.state import CrawlState
from corpus_crawler.output import ShardWriter
from corpus_crawler.fetch import RespectfulFetcher
from corpus_crawler.extract import make_records
from corpus_crawler.text import normalize_hostname

import aiohttp
from datasets import load_dataset
import pyarrow as pa
import pyarrow.parquet as pq

# ==============================================================================
# 3. CẤU HÌNH THÔNG SỐ PHÂN LÔ & TIME GUARD
# ==============================================================================
OWNER = "b"
OFFSET = 0                        # Thay đổi sau mỗi lần chạy (0, 35000, 70000...)
BATCH_SIZE = 50                # Số URL cào trong phiên này
MAX_RUNTIME_HOURS = 10.0          # Time Guard: dừng an toàn sau đúng 10 tiếng

START_TIME = time.time()
MAX_RUNTIME_SECONDS = MAX_RUNTIME_HOURS * 3600

# Thư mục dữ liệu gốc của dự án
CORPUS_ROOT = Path("corpus").resolve()
(CORPUS_ROOT / "viz" / "docs" / OWNER).mkdir(parents=True, exist_ok=True)
(CORPUS_ROOT / "viz" / "crawl_log" / OWNER).mkdir(parents=True, exist_ok=True)
(CORPUS_ROOT / "viz" / "manifest" / OWNER).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 4. DANH SÁCH 49-51 DOMAIN NHỎ CỦA MEMBER B (ĐÃ LOẠI 2 DOMAIN KHỦNG)
# ------------------------------------------------------------------------------
DOMAINS_B_BIG = frozenset({
    "120ask.com", "familydoctor.com.cn"
})

# ==============================================================================
# 5. KHỞI TẠO ĐỐI TƯỢNG SETTINGS CỦA CORPUS_CRAWLER
# ==============================================================================
config_file = CRAWLER_DIR / "config.toml"
settings = load_settings(config_file if config_file.exists() else None)

# Ghi đè cấu hình để chạy riêng cho 49 domain nhỏ của Member B
settings = replace(
    settings,
    owner=OWNER,
    corpus_root=CORPUS_ROOT,
    domains=DOMAINS_B_BIG,
    global_concurrency=32,
    per_domain_concurrency=1,
    requests_per_domain=1.0,
    shard_docs=20000,
    progress_seconds=5,
    user_agent="R2AI2026CorpusResearch/1.0 (Respectful multi-domain crawler - Member B)"
)

# Khởi tạo CrawlState (Database SQLite WAL để checkpoint chống mất dữ liệu)
state_db_path = CORPUS_ROOT / "viz" / f"state-{OWNER}-small.sqlite3"
state = CrawlState(state_db_path)

# ==============================================================================
# 6. TẢI DỮ LIỆU TỪ HUGGING FACE VÀ NẠP VÀO CRAWLSTATE THEO BATCH
# ==============================================================================
if not state.has_registered_urls():
    print("[*] Đang tải corpus từ Hugging Face: AIGuruTinix/ViBioMIR...")
    hf_dataset = load_dataset("AIGuruTinix/ViBioMIR", "corpus", split="train")
    
    print("[*] Đang lọc URL thuộc 2 domain lớn...")
    all_big_urls = []
    for row in hf_dataset:
        d = normalize_hostname(row["url"])
        if d in DOMAINS_B_BIG:
            all_big_urls.append((int(row["id"]), str(row["url"]), str(d)))
            
    print(f"[✓] Tổng số URL từ các domain lớn: {len(all_big_urls):,}")
    
    # Cắt phân lô theo OFFSET & BATCH_SIZE
    target_batch = all_big_urls[OFFSET : OFFSET + BATCH_SIZE]
    print(f"[*] Nạp batch #{OFFSET:,} -> #{OFFSET + len(target_batch):,} vào State DB...")
    
    # Nạp vào database checkpoint bằng hàm nội bộ của state.py
    state._insert_source_batch(target_batch)
    print(f"[✓] Đã nạp thành công {len(target_batch):,} URLs vào {state_db_path.name}")
else:
    print(f"[*] Tiếp tục từ State DB đã có: {state_db_path.name}")

total_urls, done_urls, pending_urls = state.totals()
print(f"-> Tổng URL batch: {total_urls:,} | Đã hoàn thành: {done_urls:,} | Còn lại: {pending_urls:,}")

# Khởi tạo ShardWriter chuẩn từ src/corpus_crawler/output.py
writer = ShardWriter(settings, state)

# ==============================================================================
# 7. ENGINE CHẠY ĐỒNG THỜI KÈM TIME GUARD 10 GIỜ (TẬN DỤNG FETCH.PY & EXTRACT.PY)
# ==============================================================================
async def run_batch_with_timeguard():
    connector = aiohttp.TCPConnector(limit=settings.global_concurrency)
    timeout = aiohttp.ClientTimeout(total=settings.timeout_seconds)
    headers = {"User-Agent": settings.user_agent}
    
    stop_event = asyncio.Event()
    last_report = time.monotonic()
    
    # Sử dụng RespectfulFetcher chính gốc từ src/corpus_crawler/fetch.py
    # (Đã tích hợp sẵn Protego đọc robots.txt và lock giãn cách 1s/domain)
    async with aiohttp.ClientSession(connector=connector, timeout=timeout, headers=headers) as session:
        fetcher = RespectfulFetcher(settings, session)

        async def worker():
            nonlocal last_report
            while not stop_event.is_set():
                # KIỂM TRA TIME GUARD (10 TIẾNG)
                if (time.time() - START_TIME) >= MAX_RUNTIME_SECONDS:
                    print(f"\n[!] TIME GUARD kích hoạt: Đã đạt mốc {MAX_RUNTIME_HOURS} tiếng! Dừng mềm để đóng gói an toàn.")
                    stop_event.set()
                    return

                # Lấy URL tiếp theo từ hàng đợi SQLite bằng state.claim_next()
                row = state.claim_next()
                if row is None:
                    if state.pending_count() == 0:
                        return
                    await asyncio.sleep(0.05)
                    continue

                # 1. Tải trang bằng fetcher.fetch() từ src/fetch.py
                result = await fetcher.fetch(str(row["url"]), str(row["domain"]))

                # 2. Bóc tách và chuẩn hóa bằng make_records() từ src/extract.py
                log_record, doc_record = await asyncio.to_thread(
                    make_records,
                    int(row["id"]),
                    str(row["url"]),
                    str(row["domain"]),
                    result,
                    settings,
                )

                # 3. Lưu kết quả và dedupe bằng state.save_result() từ src/state.py
                state.save_result(int(row["id"]), result.attempts, log_record, doc_record)

                # 4. Xuất shard khi đủ số lượng bằng writer.export_ready() từ src/output.py
                writer.export_ready()

                # Báo cáo tiến độ ra màn hình
                now = time.monotonic()
                if now - last_report >= settings.progress_seconds:
                    total, done, pending = state.totals()
                    elapsed = max(time.time() - START_TIME, 0.001)
                    rate = done / elapsed
                    eta_m = (pending / rate / 60) if rate > 0 else 0
                    print(
                        f"\rTiến độ: {done:,}/{total:,} ({done/total*100:.1f}%) | "
                        f"{rate:.1f} URL/s | Còn lại: {pending:,} | ETA: {eta_m:.1f} phút",
                        end="",
                        flush=True
                    )
                    last_report = now

        # Chạy 32 workers song song
        workers = [asyncio.create_task(worker()) for _ in range(settings.global_concurrency)]
        try:
            await asyncio.gather(*workers)
        except Exception as e:
            stop_event.set()
            raise e
        finally:
            # Ép đóng shard dở dang và ghi manifest đầy đủ
            writer.export_ready(force=True)

# ------------------------------------------------------------------------------
# 8. THỰC THI VÀ ĐÓNG GÓI ZIP OUTPUT
# ------------------------------------------------------------------------------
print("\n[*] Khởi động tiến trình cào dữ liệu...")
try:
    asyncio.run(run_batch_with_timeguard())
finally:
    state.close()

print("\n\n" + "="*70)
print(f"[✓] CÀO DỮ LIỆU HOÀN TẤT CHO BATCH #{OFFSET}!")
# print(f"• Thống kê trạng thái: {state.completed_counts()}")

# Tự động nén toàn bộ thư mục corpus thành file zip theo batch
zip_filename = f"/kaggle/working/corpus_b_big_offset_{OFFSET}.zip"
print(f"[*] Đang nén kết quả thành: {zip_filename}...")
os.system(f"zip -r -q {zip_filename} corpus/")
print(f"[✓] HOÀN TẤT AN TOÀN! Tệp '{zip_filename}' đã sẵn sàng trong thư mục làm việc.")

d:\DevTools\Python3-13\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] Đang tải corpus từ Hugging Face: AIGuruTinix/ViBioMIR...


[*] Đang lọc URL thuộc 2 domain lớn...
[✓] Tổng số URL từ các domain lớn: 1,365,932
[*] Nạp batch #0 -> #50 vào State DB...
[✓] Đã nạp thành công 50 URLs vào state-b-small.sqlite3
-> Tổng URL batch: 50 | Đã hoàn thành: 0 | Còn lại: 50

[*] Khởi động tiến trình cào dữ liệu...
Tiến độ: 50/50 (100.0%) | 0.2 URL/s | Còn lại: 0 | ETA: 0.0 phútt

[✓] CÀO DỮ LIỆU HOÀN TẤT CHO BATCH #0!


ProgrammingError: Cannot operate on a closed database.

In [ ]:
# import os
# import time
# import json
# import asyncio
# import hashlib
# import unicodedata
# from datetime import datetime, timezone, timedelta
# from urllib.parse import urlsplit

# import aiohttp
# import trafilatura
# import zstandard as zstd
# import pyarrow as pa
# import pyarrow.parquet as pq
# from datasets import load_dataset

# # ==============================================================================
# # 2. CẤU HÌNH BATCH CHO TỪNG NOTEBOOK (TÙY CHỈNH THEO Ý BẠN)
# # ==============================================================================
# OWNER = "b"
# TARGET_DOMAIN = "120ask.com"      # Hoặc "familydoctor.com.cn"
# OFFSET = 0                        # Vị trí bắt đầu cào (Notebook 2 đổi thành 35000, Notebook 3 đổi thành 70000...)
# BATCH_SIZE = 35000                # Số URL cào trong phiên này (35k URL ~ 9.5 tiếng, cực an toàn)
# MAX_RUNTIME_HOURS = 10.8          # Bộ đếm an toàn: tự dừng sau 10.8 tiếng để kịp nén file zip

# START_TIME = time.time()
# MAX_RUNTIME_SECONDS = MAX_RUNTIME_HOURS * 3600

# KAGGLE_OUT = "/kaggle/working"
# DIR_DOCS = f"{KAGGLE_OUT}/corpus/viz/docs/{OWNER}"
# DIR_LOGS = f"{KAGGLE_OUT}/corpus/viz/crawl_log/{OWNER}"
# DIR_MANIFEST = f"{KAGGLE_OUT}/corpus/viz/manifest/{OWNER}"

# for d in [DIR_DOCS, DIR_LOGS, DIR_MANIFEST]:
#     os.makedirs(d, exist_ok=True)

# # ==============================================================================
# # 3. TẢI VÀ TRÍCH XUẤT ĐÚNG PHÂN ĐOẠN (SLICING) TỪ HUGGING FACE
# # ==============================================================================
# print(f"[*] Đang tải danh sách URL của domain [{TARGET_DOMAIN}] từ Hugging Face...")
# hf_dataset = load_dataset("AIGuruTinix/ViBioMIR", "corpus", split="train")

# def get_clean_host(u):
#     try:
#         h = urlsplit(u).hostname or ""
#         h = h.lower().rstrip(".")
#         return h[4:] if h.startswith("www.") else h
#     except:
#         return ""

# # Lọc chỉ lấy domain mục tiêu
# domain_urls = [
#     {"id": int(r["id"]), "url": r["url"]} 
#     for r in hf_dataset if get_clean_host(r["url"]) == TARGET_DOMAIN
# ]

# total_domain_urls = len(domain_urls)
# print(f"[✓] Tổng số URL của {TARGET_DOMAIN}: {total_domain_urls:,}")

# # Cắt lấy phân đoạn của Notebook này
# target_batch = domain_urls[OFFSET : OFFSET + BATCH_SIZE]
# print(f"[*] Notebook này sẽ cào từ URL index #{OFFSET:,} đến #{OFFSET + len(target_batch):,} (Tổng: {len(target_batch):,} URLs)")

# # ==============================================================================
# # 3. CHUẨN HÓA VĂN BẢN VÀ BỘ GHI SHARD (ZSTD ATOMIC WRITER)
# # ==============================================================================
# def normalize_text(raw_text: str) -> str:
#     if not raw_text: return ""
#     text = unicodedata.normalize("NFC", raw_text).replace("\r\n", "\n").replace("\r", "\n")
#     text = "".join(ch for ch in text if ch in ("\n", "\t") or not unicodedata.category(ch).startswith("C"))
#     lines = [" ".join(l.split()) for l in text.split("\n")]
#     res = "\n".join(lines)
#     while "\n\n\n" in res: res = res.replace("\n\n\n", "\n\n")
#     return res.strip()

# class BatchShardWriter:
#     def __init__(self, start_seq=1, shard_size=20000):
#         self.shard_size = shard_size
#         self.seq = start_seq
#         self.count = 0
#         self.cctx = zstd.ZstdCompressor(level=3)
#         self._new_shard()

#     def _new_shard(self):
#         self.shard_name = f"viz-{OWNER}-{TARGET_DOMAIN.replace('.', '_')}-{self.seq:05d}.jsonl.zst"
#         self.final_path = os.path.join(DIR_DOCS, self.shard_name)
#         self.temp_path = self.final_path + ".tmp"
#         self.f_out = open(self.temp_path, "wb")
#         self.writer = self.cctx.stream_writer(self.f_out)
#         self.count = 0

#     def write(self, doc):
#         line = (json.dumps(doc, ensure_ascii=False) + "\n").encode("utf-8")
#         self.writer.write(line)
#         self.count += 1
#         if self.count >= self.shard_size:
#             self.close()
#             self.seq += 1
#             self._new_shard()

#     def close(self):
#         if self.writer:
#             self.writer.close()
#             self.f_out.close()
#         if self.count > 0 and os.path.exists(self.temp_path):
#             os.rename(self.temp_path, self.final_path)
#             hasher = hashlib.sha256()
#             with open(self.final_path, "rb") as f:
#                 while chunk := f.read(65536): hasher.update(chunk)
#             manifest = {
#                 "shard": self.shard_name, "owner": OWNER, "n_docs": self.count,
#                 "sha256": hasher.hexdigest(), "created_at": datetime.now(timezone(timedelta(hours=7))).isoformat()
#             }
#             with open(os.path.join(DIR_MANIFEST, f"{self.shard_name}.manifest.json"), "w") as mf:
#                 json.dump(manifest, mf, indent=2)
#             print(f"\n[✓ ĐÃ ĐÓNG SHARD] {self.shard_name} ({self.count:,} docs)")

# # ==============================================================================
# # 4. ENGINE CÀO BỀN VỮNG KÈM TIME GUARD
# # ==============================================================================
# async def run_batch_crawl():
#     start_seq = (OFFSET // 20000) + 1
#     writer = BatchShardWriter(start_seq=start_seq, shard_size=20000)
    
#     headers = {"User-Agent": "R2AI2026CorpusResearch/1.0 (Kaggle respectful batch crawler)"}
#     connector = aiohttp.TCPConnector(limit=8)
#     timeout = aiohttp.ClientTimeout(total=25)
    
#     next_call_time = 0.0
#     completed = 0
#     total = len(target_batch)
#     rate_delay = 1.0  # Giữ tốc độ an toàn 1.0 giây / request cho domain lớn

#     async with aiohttp.ClientSession(connector=connector, timeout=timeout, headers=headers) as session:
#         for item in target_batch:
#             # KIỂM TRA TIME GUARD: Nếu sắp chạm 10.8 tiếng thì tự ngắt an toàn
#             elapsed = time.time() - START_TIME
#             if elapsed >= MAX_RUNTIME_SECONDS:
#                 print(f"\n[!] CẢNH BÁO TIME GUARD: Đã chạy {elapsed/3600:.1f} tiếng. Tự dừng an toàn để nén file zip!")
#                 break

#             # Điều tiết tốc độ lịch sự với domain
#             wait = next_call_time - time.time()
#             if wait > 0:
#                 await asyncio.sleep(wait)
#             next_call_time = time.time() + rate_delay

#             status = "ok"
#             http_status = None
#             final_text = ""
#             title = ""

#             try:
#                 async with session.get(item["url"], allow_redirects=True) as resp:
#                     http_status = resp.status
#                     if resp.status == 200:
#                         html = await resp.text(errors="replace")
#                         extracted = trafilatura.extract(html, include_comments=False, include_tables=True)
#                         if extracted and len(extracted.strip()) > 0:
#                             final_text = normalize_text(extracted)
#                             meta = trafilatura.extract_metadata(html)
#                             title = normalize_text(meta.title or "") if meta else ""
#                         else:
#                             status = "empty"
#                     else:
#                         status = "http_error"
#             except asyncio.TimeoutError:
#                 status = "timeout"
#             except Exception:
#                 status = "http_error"

#             if status == "ok" and final_text and len(final_text) >= 100:
#                 doc = {
#                     "schema_version": "1.0",
#                     "uid": f"src:{item['id']}",
#                     "doc_id": str(item["id"]),
#                     "lang": "zh",
#                     "source": "btc_url",
#                     "url": item["url"],
#                     "domain": TARGET_DOMAIN,
#                     "title": title,
#                     "text": final_text,
#                     "norm_version": "n1",
#                     "passages": [{"type": "body", "start": 0, "end": len(final_text)}],
#                     "text_len": len(final_text),
#                     "content_hash": f"sha1:{hashlib.sha1(final_text.encode('utf-8')).hexdigest()}",
#                     "dup_of": None,
#                     "fetch": {"status": status, "http_status": http_status, "fetched_at": datetime.now(timezone(timedelta(hours=7))).isoformat()},
#                     "owner": OWNER
#                 }
#                 writer.write(doc)

#             completed += 1
#             if completed % 100 == 0:
#                 cur_rate = completed / max(time.time() - START_TIME, 0.001)
#                 rem_urls = total - completed
#                 eta_h = rem_urls / cur_rate / 3600 if cur_rate > 0 else 0
#                 print(f"\rCào {TARGET_DOMAIN}: {completed:,}/{total:,} ({completed/total*100:.1f}%) | {cur_rate:.2f} req/s | ETA: {eta_h:.1f}h", end="", flush=True)

#     writer.close()

# # CHẠY
# asyncio.run(run_batch_crawl())

# # ==============================================================================
# # 5. TỰ ĐỘNG NÉN FILE ZIP VÀO OUTPUT
# # ==============================================================================
# zip_filename = f"corpus_{TARGET_DOMAIN.replace('.', '_')}_offset_{OFFSET}.zip"
# print(f"\n[*] Đang nén kết quả vào {KAGGLE_OUT}/{zip_filename}...")
# !cd /kaggle/working && zip -r -q {zip_filename} corpus/
# print(f"[✓] HOÀN TẤT AN TOÀN! Tải file {zip_filename} trong tab Output.")